In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix


biased training set

In [2]:
train_size = 500
ethnicity1 = ['red' for i in range(train_size)]
decision1 = [0 for i in range(train_size)]
ethnicity2 = ['purple' for i in range(train_size)]
decision2 = [1 for i in range(train_size)]
income1 = np.random.randint(5000, 65000, size=train_size)
income2 = np.random.randint(65000, 200000, size=train_size)
data = {'ethnicity': ethnicity1+ethnicity2,
        'income': np.concatenate((income1, income2)),
        'decision': decision1+decision2}
df = pd.DataFrame(data)
df_shuffled = df.sample(frac=1).reset_index(drop=True)
df_shuffled.head()


,ethnicity,income,decision
0,red,18495,0
1,red,40952,0
2,red,59828,0
3,red,37528,0
4,purple,160356,1


general test set

In [3]:
test_size = 500
ethnicity = np.random.randint(0, 2, test_size*2)
ethnicity = ['purple' if i==0 else 'red' for i in ethnicity]
income1 = np.random.randint(5000, 65000, size=test_size)
decision1 = [0 for i in range(test_size)]
income2 = np.random.randint(65000, 200000, size=test_size)
decision2 = [1 for i in range(test_size)]
data_test = {'ethnicity': ethnicity,
        'income': np.concatenate((income1, income2)),
        'decision': decision1+decision2}
test_df = pd.DataFrame(data_test)
test_df_shuffled = test_df.sample(frac=1).reset_index(drop=True)
test_df_shuffled.head()

,ethnicity,income,decision
0,red,49926,0
1,purple,118675,1
2,red,49872,0
3,purple,197412,1
4,purple,183354,1


preprocessing

In [4]:
train_df_encoded = pd.get_dummies(df_shuffled, columns=['ethnicity'])
test_df_encoded = pd.get_dummies(test_df_shuffled, columns=['ethnicity'])

In [5]:
X_train = train_df_encoded.drop(['decision'], axis=1)

X_test = test_df_encoded.drop(['decision'], axis=1)
X_test.head()

,income,ethnicity_purple,ethnicity_red
0,49926,False,True
1,118675,True,False
2,49872,False,True
3,197412,True,False
4,183354,True,False


In [6]:

y_train = train_df_encoded['decision']
y_test = test_df_encoded['decision']
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [7]:
len(X_train_scaled)

1000

In [8]:
class LoanPredictor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)

def train(X_train, y_train):

    
    # Convert to PyTorch tensors
    X_train_tensor = torch.FloatTensor(X_train)
    y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)

    

    model = LoanPredictor(input_dim=X_train.shape[1])
    

    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters())
    
    # Training loop
    num_epochs = 100
    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train_tensor)
        loss = criterion(outputs, y_train_tensor)
        loss.backward()
        optimizer.step()
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item()}')
    return model

    



In [9]:

model = train(X_train_scaled, y_train)

Epoch [1/100], Loss: 0.6614111661911011
Epoch [2/100], Loss: 0.654342770576477
Epoch [3/100], Loss: 0.6502887010574341
Epoch [4/100], Loss: 0.6517267227172852
Epoch [5/100], Loss: 0.6482554078102112
Epoch [6/100], Loss: 0.6440421342849731
Epoch [7/100], Loss: 0.6421140432357788
Epoch [8/100], Loss: 0.6336832046508789
Epoch [9/100], Loss: 0.6328961253166199
Epoch [10/100], Loss: 0.6325278282165527
Epoch [11/100], Loss: 0.6288878321647644
Epoch [12/100], Loss: 0.6240448951721191
Epoch [13/100], Loss: 0.6221281290054321
Epoch [14/100], Loss: 0.6213106513023376
Epoch [15/100], Loss: 0.6157164573669434
Epoch [16/100], Loss: 0.612034797668457
Epoch [17/100], Loss: 0.608771562576294
Epoch [18/100], Loss: 0.6046317219734192
Epoch [19/100], Loss: 0.6028482913970947
Epoch [20/100], Loss: 0.6000410914421082
Epoch [21/100], Loss: 0.59750896692276
Epoch [22/100], Loss: 0.594518780708313
Epoch [23/100], Loss: 0.5887719988822937
Epoch [24/100], Loss: 0.5865737199783325
Epoch [25/100], Loss: 0.5843088

In [11]:
torch.save(model.state_dict(), 'loanapp.pth')

In [12]:


# Evaluate fairness
model.eval()
    # Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.FloatTensor(y_train.values).reshape(-1, 1)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.FloatTensor(y_test.values).reshape(-1, 1)
with torch.no_grad():
    test_preds = model(X_test_tensor)
    predictions = (test_preds > 0.5).float()
    train_preds = model(X_train_tensor)
    train_preds = (train_preds > 0.5).float()
    train_accuracy = accuracy_score(y_train, train_preds)
    accuracy = accuracy_score(y_test, predictions)
    conf_matrix = confusion_matrix(y_test, predictions)
    
    print(f"Model Accuracy: {accuracy}")
    print(f'model accuracy on training data: {train_accuracy}')
    print("Confusion Matrix:\n", conf_matrix)

Model Accuracy: 0.506
model accuracy on training data: 1.0
Confusion Matrix:
 [[255 245]
 [249 251]]


In [13]:
class InputTransformer(nn.Module):
    def __init__(self, input_dim=3):
        super(InputTransformer, self).__init__()
        
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, input_dim),
            nn.Tanh()
        )
    
    def forward(self, x):
        y = self.net(x)
        return self.net(x)

In [14]:
class CombinedModel(nn.Module):
    def __init__(self, loan_predictor, transformer):
        super(CombinedModel, self).__init__()
        self.loan_predictor = loan_predictor
        self.transformer = transformer
    
    
        # Freeze the loan_predictor's parameters
        for param in self.loan_predictor.parameters():
            param.requires_grad = False
    
    def forward(self, x):
        transformed_input = self.transformer(x)
        # with torch.no_grad():
        output = self.loan_predictor(transformed_input)
        
        return {'transformed_input':transformed_input,
                'output': output}

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [16]:
# model.eval()
# for param in model.parameters():
#     param.requires_grad = False

input_loss_weight = 0.1
input_transformer = InputTransformer(input_dim=3).to(device)
combined_model = CombinedModel(model, input_transformer)
optimizer = torch.optim.Adam(combined_model.transformer.parameters(), lr=0.0005)
criterion = nn.MSELoss()
mse_loss = nn.MSELoss()
for param in input_transformer.parameters():
    param.requires_grad = True

num_epochs = 50
for epoch in range(num_epochs):
    combined_model.train()
    optimizer.zero_grad()
    combined_outputs = combined_model(X_test_tensor)
    loss_prediction = criterion(combined_outputs['output'], y_test_tensor)
    loss_inputchange = mse_loss(X_test_tensor, combined_outputs['transformed_input'])
    total_loss = input_loss_weight * loss_inputchange + loss_prediction
    total_loss.backward()
    #loss_prediction.backward()
    optimizer.step()
    print(f'Epoch [{epoch+1}],= (predloss: {loss_prediction} + inputloss: {loss_inputchange}')

Epoch [1],= (predloss: 0.2431606650352478 + inputloss: 1.1735023260116577
Epoch [2],= (predloss: 0.24153180420398712 + inputloss: 1.1322585344314575
Epoch [3],= (predloss: 0.23748278617858887 + inputloss: 1.0847582817077637
Epoch [4],= (predloss: 0.23761960864067078 + inputloss: 1.0710338354110718
Epoch [5],= (predloss: 0.233775794506073 + inputloss: 1.0184743404388428
Epoch [6],= (predloss: 0.23232363164424896 + inputloss: 0.9820567965507507
Epoch [7],= (predloss: 0.22828872501850128 + inputloss: 0.9613790512084961
Epoch [8],= (predloss: 0.2278013527393341 + inputloss: 0.9208123087882996
Epoch [9],= (predloss: 0.22426079213619232 + inputloss: 0.8979296088218689
Epoch [10],= (predloss: 0.22589747607707977 + inputloss: 0.8798879981040955
Epoch [11],= (predloss: 0.22110141813755035 + inputloss: 0.8591958284378052
Epoch [12],= (predloss: 0.22034786641597748 + inputloss: 0.8384158611297607
Epoch [13],= (predloss: 0.21787533164024353 + inputloss: 0.8151117563247681
Epoch [14],= (predloss: 0

In [17]:
with torch.no_grad():
    test_preds = model(X_test_tensor)
    combined_test_preds = combined_model(X_test_tensor)
    print(combined_test_preds['transformed_input'])
    predictions = (test_preds > 0.5).float()

    combined_test_preds = (combined_test_preds['output'] > 0.5).float()
    combined_acc = accuracy_score(y_test, combined_test_preds)
    accuracy = accuracy_score(y_test, predictions)
    conf_matrix = confusion_matrix(y_test, combined_test_preds)
    
    print(f"Model Accuracy: {accuracy}")
    print(f'combined accuracy: {combined_acc}')
    print("Confusion Matrix:\n", conf_matrix)

tensor([[-0.4194, -0.4793,  0.6926],
        [ 0.5777,  0.6433, -0.8075],
        [-0.5812, -0.8025,  0.3865],
        ...,
        [-0.7447,  0.6054,  0.8805],
        [ 0.8643,  0.9386, -0.8244],
        [-0.7436, -0.7442,  0.6164]])
Model Accuracy: 0.549
combined accuracy: 0.735
Confusion Matrix:
 [[457  43]
 [222 278]]


In [18]:
latent_transformer = LoanPredictor(input_dim=3)
latent_transformer.load_state_dict(torch.load('loanapp.pth'))

<All keys matched successfully>

In [20]:
def compare(model1, model2):
    for p1, p2 in zip(model1.parameters(), model2.parameters()):
        if p1.data.ne(p2.data).sum() > 0:
            return False
    return True

In [ ]:
print(co)

In [21]:
compare(model, latent_transformer)

True

In [26]:
def evaluate_and_find_improved_cases(combined_model, original_model, X_test_tensor, y_test_tensor, num_cases=5):
    combined_model.eval()
    original_model.eval()
    
    # Get predictions
    with torch.no_grad():
        # Original model predictions
        orig_preds = original_model(X_test_tensor)
        orig_labels = (orig_preds > 0.5).float()
        
        # Combined model predictions
        combined_outputs = combined_model(X_test_tensor)
        combined_preds = (combined_outputs['output'] > 0.5).float()
        
        # Find improved cases
        improved_mask = (orig_labels != y_test_tensor) & (combined_preds == y_test_tensor)
        #print(improved_mask)
        # Get indices of improved cases
        improved_indices = torch.where(improved_mask)[0]
        
        # Print overall accuracy
        orig_accuracy = (orig_labels == y_test_tensor).float().mean()
        combined_accuracy = (combined_preds == y_test_tensor).float().mean()
        
        print(f"Original Model Accuracy: {orig_accuracy.item():.4f}")
        print(f"Combined Model Accuracy: {combined_accuracy.item():.4f}")
        
        # Showcase improved cases
        print("\nImproved Cases:")
        for i, idx in enumerate(improved_indices[:num_cases]):
            print(f"\nCase {i+1}:")
            print(f"Original Prediction: {orig_labels[idx].item()}")
            print(f"Correct Label: {y_test_tensor[idx].item()}")
            print(f"Combined Model Prediction: {combined_preds[idx].item()}")
            
            # Optional: visualize input transformations if needed
            original_input = X_test_tensor[idx]
            transformed_input = combined_model(original_input.unsqueeze(0))['transformed_input']
            
            print("\nOriginal Input:", original_input)
            print("Transformed Input:", transformed_input)
        
        return improved_indices[:num_cases]

# Usage
improved_cases = evaluate_and_find_improved_cases(
    combined_model, 
    model, 
    X_test_tensor, 
    y_test_tensor
)

Original Model Accuracy: 0.5060
Combined Model Accuracy: 0.7760

Improved Cases:

Case 1:
Original Prediction: 1.0
Correct Label: 0.0
Combined Model Prediction: 0.0

Original Input: tensor([-0.7451,  1.0000, -1.0000])
Transformed Input: tensor([[-0.6917,  0.1569, -0.1948]])

Case 2:
Original Prediction: 1.0
Correct Label: 0.0
Combined Model Prediction: 0.0

Original Input: tensor([-1.3117,  1.0000, -1.0000])
Transformed Input: tensor([[-0.8535,  0.0837,  0.0613]])

Case 3:
Original Prediction: 1.0
Correct Label: 0.0
Combined Model Prediction: 0.0

Original Input: tensor([-1.1671,  1.0000, -1.0000])
Transformed Input: tensor([[-0.8178,  0.1011, -0.0066]])

Case 4:
Original Prediction: 1.0
Correct Label: 0.0
Combined Model Prediction: 0.0

Original Input: tensor([-0.7355,  1.0000, -1.0000])
Transformed Input: tensor([[-0.6883,  0.1581, -0.1990]])

Case 5:
Original Prediction: 0.0
Correct Label: 1.0
Combined Model Prediction: 1.0

Original Input: tensor([ 1.8649, -1.0000,  1.0000])
Transf